# Appendix

This notebook contains additional code used to generate some ancillary inputs for the demos in the other notebooks.

## Compute $\overline{SIF_j}$ and $\overline{\Delta SIF_j}$ for a Region of Interest

To derive the Z-scores and RCI for the SIF data used in the first notebook, we need to determine mean SIF over the entire time range of our source data for each j-th 8-day window. For completeness, we will download the entire GOSIF dataset from 2001-2020 and compute the mean value over the region of interest from the notebook, saving the result in a CSV. The download step will take about 20 minutes

In [ ]:
import os
from tqdm.notebook import tqdm
from download import download_unpack_gosif

dates: list[tuple[int, int]] = []
for year in range(2001, 2021):
    for doy in range(1, 365, 8):
        dates.append((year, doy))

output_dir = "data/gosif"
os.makedirs(output_dir, exist_ok=True)

gosif_geotiffs: list[str] = []
for date_tuple in tqdm(dates, desc="Downloading granules"):
    fname = download_unpack_gosif(
        date_tuple[0],
        day=date_tuple[1],
        output_dir=output_dir,
        verbose=False
    )
    if fname:
        gosif_geotiffs.append(fname)

In [ ]:
from datetime import datetime
from glob import glob
import numpy as np
import rasterio
from rasterio.windows import from_bounds

output_dir = "data/gosif"
gosif_geotiffs = sorted(glob(f"{output_dir}/GOSIF_*.tif"))

# Set the filename to use for the time series
time_series_fname = "northern_great_plains_2017_sif.csv"

# Northern Great Plains region of interest
# 45.00°–50.00°N, 106.00°–111.00°W
west, south, east, north = -111.0, 45.0, -106.0, 50.0

# The threshold and scale factor parameters come from the documentation: https://data.globalecology.unh.edu/data/GOSIF_v2/Fair_Data_Use_Policy_and_Readme_GOSIF_v2.pdf
# 32767 = water bodies, 32766 = ice/snow
gosif_data_thresh = 32765
# This value tells our code the conversion between pixel values in the GeoTIFF images to units of W/m^2/sr/μm
gosif_scale_factor = 0.0001

dates: list[datetime] = []
sif_means: list[float] = []
delta_sif: list[float] = []
spatial_mean_prev = 0.0
for geotiff in gosif_geotiffs:
    # Parse the date from the filename, e.g. GOSIF_2017073.tif = DOY 73
    doy = int(os.path.splitext(os.path.basename(geotiff))[0][-3:])
    dates.append(datetime.strptime(f"{year}{doy:03d}", "%Y%j"))

    with rasterio.open(geotiff) as src:
        window = from_bounds(west, south, east, north, src.transform)
        data = src.read(1, window=window).astype(float)
        data[data > gosif_data_thresh] = np.nan
        spatial_mean = float(np.nanmean(data)) * gosif_scale_factor
        dsif = spatial_mean - spatial_mean_prev
        sif_means.append(spatial_mean)
        delta_sif.append(dsif)